In [0]:
%run ../Databricks-Bookstore-Engineering/Copy-Datasets



In [0]:
%sql
drop table bronze

In [0]:
# %run ../Databricks-Bookstore-Engineering/Copy-Datasets


current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
dataset_bookstore = f"/Volumes/{current_catalog}/bookstore_eng_pro/dataset"
print(dataset_bookstore)

In [0]:
files = dbutils.fs.ls(f"{dataset_bookstore}/kafka-raw")
display(files)

In [0]:
df_raw = spark.read.json(f"{dataset_bookstore}/kafka-raw")
display(df_raw)

In [0]:
from pyspark.sql import functions as F

def process_bronze():
    schema = "key BINARY, value BINARY, topic STRING, partition LONG, offset LONG, timestamp LONG"

    query = (spark.readStream
                        .format("cloudFiles")
                        .option("cloudFiles.format","json")
                        .schema(schema)
                        .load(f"{dataset_bookstore}/kafka-raw")
                        .withColumn("timestamp", (F.col("timestamp")/1000).cast("timestamp"))
                        .withColumn("year_month", F.date_format("timestamp", "yyyy-MM"))
                    .writeStream
                        .option("checkpointLocation", "/Volumes/certspace/bookstore_eng_pro/checkpoints/bronze")
                        .option("mergeSchema", True)
                        .partitionBy("topic", "year_month")
                        .trigger(availableNow=True)
                        .table("bronze"))
    
    query.awaitTermination()


    


In [0]:
process_bronze()

In [0]:
batch_df = spark.table("bronze")
display(batch_df)

In [0]:
%sql
SELECT * FROM bronze

In [0]:
%sql
SELECT distinct (topic) from bronze

In [0]:
bookstore.load_new_data()

In [0]:
process_bronze()

In [0]:
%sql
SELECT count(*) from bronze